# 03 — Leakage-Free Feature Engineering
**Goal:** Build one row per match using only historical World Cup games played before each match's date. This is **not current national-team form** (qualifiers, friendlies, FIFA rankings are not available in the supplied sources). Run notebook 01 first.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data' / 'raw').exists():
    raise FileNotFoundError('Open this notebook with working directory at the project root or notebooks folder.')
sys.path.insert(0, str(PROJECT_ROOT))
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_REPORTS = PROJECT_ROOT / 'data' / 'reports'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_REPORTS.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)

Project root: C:\Users\User\Documents\fifa_world_cup_intelligence


In [2]:
import pandas as pd
from IPython.display import display
from src.worldcup_data import create_prematch_features, FEATURE_COLUMNS
matches=pd.read_csv(DATA_PROCESSED/'matches_clean.csv',parse_dates=['date'])
# CSV booleans should normally reload as bool; force explicit numeric eligibility.
for col in ['ml_eligible','ambiguous_flat_knockout']:
    matches[col]=matches[col].astype(str).str.lower().eq('true')
features=create_prematch_features(matches)
print('Features shape:',features.shape)
display(features.tail(4))

Features shape: (1069, 23)


,match_key,year,date,team_1,team_2,round,goals_1,goals_2,result_90,ml_eligible,...,team_2_win_rate_last5,team_1_goals_for_last5,team_2_goals_for_last5,team_1_goals_against_last5,team_2_goals_against_last5,win_rate_diff,goals_for_diff,goals_against_diff,head_to_head_matches,head_to_head_team1_win_rate
1065,2026|2026-07-14|France|Spain,2026,2026-07-14,France,Spain,Semi-final,0.0,2.0,team_2_win,True,...,1.0,2.6,2.2,0.2,0.2,0.0,0.4,0.0,1,1.000000
1066,2026|2026-07-15|Argentina|England,2026,2026-07-15,England,Argentina,Semi-final,1.0,2.0,team_2_win,True,...,0.6,1.6,2.0,0.8,1.0,0.0,-0.4,-0.2,3,0.666667
1067,2026|2026-07-18|England|France,2026,2026-07-18,France,England,Match for third place,4.0,6.0,team_2_win,True,...,0.6,2.0,1.8,0.6,1.2,0.2,0.2,-0.6,3,0.333333
1068,2026|2026-07-19|Argentina|Spain,2026,2026-07-19,Spain,Argentina,Final,0.0,0.0,draw,True,...,0.6,1.8,2.0,0.2,1.2,0.4,-0.2,-1.0,1,0.000000


## A. Inspect engineered features
Features include a five-match World Cup win rate, average goals scored and conceded, number of prior World Cup matches and earlier head-to-head results. Both teams’ features are calculated before results from the same calendar day enter history.

In [3]:
display(features[FEATURE_COLUMNS].describe().T)
print('Missing feature values (expected for teams with no recorded prior matches):')
display(features[FEATURE_COLUMNS].isna().sum().sort_values(ascending=False))
print('Label distribution:')
display(features.loc[features['ml_eligible'],'result_90'].value_counts())

,count,mean,std,min,25%,50%,75%,max
team_1_prev_matches,1069.0,22.304958,19.358306,0.0,7.0,17.0,33.00,89.000000
team_2_prev_matches,1069.0,17.507951,16.568244,0.0,5.0,13.0,25.00,87.000000
team_1_win_rate_last5,1031.0,0.419027,0.256339,0.0,0.2,0.4,0.60,1.000000
team_2_win_rate_last5,1015.0,0.346026,0.258752,0.0,0.2,0.4,0.60,1.000000
team_1_goals_for_last5,1031.0,1.525994,0.843235,0.0,1.0,1.4,2.00,7.000000
team_2_goals_for_last5,1015.0,1.325665,0.784168,0.0,0.8,1.2,1.80,5.400000
team_1_goals_against_last5,1031.0,1.197882,0.779659,0.0,0.6,1.0,1.60,6.333333
team_2_goals_against_last5,1015.0,1.388227,0.918887,0.0,0.8,1.2,1.80,9.000000
win_rate_diff,986.0,0.074527,0.320138,-1.0,-0.2,0.0,0.25,1.000000
goals_for_diff,986.0,0.199256,1.012601,-3.0,-0.4,0.2,0.80,4.800000


Missing feature values (expected for teams with no recorded prior matches):


head_to_head_team1_win_rate    755
goals_against_diff              83
goals_for_diff                  83
win_rate_diff                   83
team_2_win_rate_last5           54
team_2_goals_for_last5          54
team_2_goals_against_last5      54
team_1_win_rate_last5           38
team_1_goals_for_last5          38
team_1_goals_against_last5      38
team_1_prev_matches              0
team_2_prev_matches              0
head_to_head_matches             0
dtype: int64

Label distribution:


result_90
team_1_win    448
draw          274
team_2_win    229
Name: count, dtype: int64

## B. Leakage check: first World Cup appearance of a team
A team’s first recorded match should have zero earlier matches, and its average goals/win rate should be missing rather than obtained from the future.

In [4]:
new_teams=features[(features['team_1_prev_matches']==0)|(features['team_2_prev_matches']==0)]
display(new_teams[['year','date','team_1','team_2',
                   'team_1_prev_matches','team_2_prev_matches',
                   'team_1_win_rate_last5','team_2_win_rate_last5']].head(12))
assert features['match_key'].is_unique, 'Duplicate match keys detected; inspect source data'
assert (features['team_1_prev_matches']>=0).all()
assert (features['team_2_prev_matches']>=0).all()

,year,date,team_1,team_2,team_1_prev_matches,team_2_prev_matches,team_1_win_rate_last5,team_2_win_rate_last5
0,1930,1930-07-13,United States,Belgium,0,0,NaN,NaN
1,1930,1930-07-13,France,Mexico,0,0,NaN,NaN
2,1930,1930-07-14,Yugoslavia,Brazil,0,0,NaN,NaN
3,1930,1930-07-14,Romania,Peru,0,0,NaN,NaN
4,1930,1930-07-15,Argentina,France,0,1,NaN,1.000000
5,1930,1930-07-16,Chile,Mexico,0,1,NaN,0.000000
6,1930,1930-07-17,Yugoslavia,Bolivia,1,0,1.0,NaN
7,1930,1930-07-17,United States,Paraguay,1,0,1.0,NaN
8,1930,1930-07-18,Uruguay,Peru,0,1,NaN,0.000000
18,1934,1934-05-27,Sweden,Argentina,0,5,NaN,0.800000


## C. Create the modeling table
Use `year >= 2002` for initial experiments but retain 1930–1998 history for constructing earlier features. Our targets are the **90-minute match outcomes**, not shootout advancement. Exclude ambiguous flat-score knockout records.

In [5]:
model_df=features[(features['year']>=2002)&(features['ml_eligible'])].copy()
model_df=model_df.dropna(subset=['result_90'])
print('ML matches by tournament:')
display(model_df.groupby('year').size().rename('rows').to_frame())
print('Model feature columns:',FEATURE_COLUMNS)
model_df.to_csv(DATA_PROCESSED/'prematch_features.csv',index=False,date_format='%Y-%m-%d')
print('Saved:',DATA_PROCESSED/'prematch_features.csv')

ML matches by tournament:


,rows
year,
2002,53
2006,64
2010,52
2014,64
2018,64
2022,64
2026,104


Model feature columns: ['team_1_prev_matches', 'team_2_prev_matches', 'team_1_win_rate_last5', 'team_2_win_rate_last5', 'team_1_goals_for_last5', 'team_2_goals_for_last5', 'team_1_goals_against_last5', 'team_2_goals_against_last5', 'win_rate_diff', 'goals_for_diff', 'goals_against_diff', 'head_to_head_matches', 'head_to_head_team1_win_rate']
Saved: C:\Users\User\Documents\fifa_world_cup_intelligence\data\processed\prematch_features.csv
